# 第 2 课阶段整理：让订单检查可以重复运行

本 Notebook 从原始 CSV 开始，不依赖第一份 Notebook 的变量。保留全部订单，只转换类型、增加标记和第一版指标。

本次重点：理解函数的输入、工作副本和返回值。代码规则与前面逐步执行的版本保持一致。

In [3]:
from pathlib import Path
import sys
import pandas as pd

project_dir = Path(r'D:\数据分析\olist-operations-analytics')
data_dir = project_dir.parent / 'archive'

# 告诉 Python 去项目的 src 文件夹寻找我们写的模块。
if str(project_dir / 'src') not in sys.path:
    sys.path.insert(0, str(project_dir / 'src'))

from orders_cleaning import clean_orders, summarize_metrics, REVIEW_FLAG_COLS

## 输入和返回值

`raw_orders` 是原始输入。`clean_orders(raw_orders)` 调用函数，函数内部使用副本，返回新的 DataFrame。把返回值赋给 `orders_v1`，后面才能继续使用。

需要查看完整函数时，打开 `src/orders_cleaning.py`。

In [4]:
raw_orders = pd.read_csv(
    data_dir / 'olist_orders_dataset.csv',
    dtype=str, encoding='utf-8-sig',
    keep_default_na=False, na_values=['']
)
orders_v1 = clean_orders(raw_orders)
print('输入行数：', len(raw_orders))
print('输出行数：', len(orders_v1))
print('原始下单时间类型：', raw_orders['order_purchase_timestamp'].dtype)
print('整理后下单时间类型：', orders_v1['order_purchase_timestamp'].dtype)

输入行数： 99441
输出行数： 99441
原始下单时间类型： object
整理后下单时间类型： datetime64[ns]


In [5]:
# 同一段函数应复现课堂的六条标记。
orders_v1[REVIEW_FLAG_COLS].sum()

flag_delivered_missing_date          8
flag_delivered_missing_approval     14
flag_delivered_missing_carrier       2
flag_carrier_before_purchase       166
flag_delivery_before_carrier        23
flag_canceled_has_delivery_date      6
dtype: int64

In [6]:
print('规则触发次数：', orders_v1[REVIEW_FLAG_COLS].sum().sum())
print('待核查订单数：', orders_v1['flag_needs_review'].sum())
print('每笔订单触发规则数的分布：')
orders_v1['review_rule_count'].value_counts().sort_index()

规则触发次数： 219
待核查订单数： 218
每笔订单触发规则数的分布：


review_rule_count
0    99223
1      217
2        1
Name: count, dtype: int64

In [7]:
# 为什么触发 219 次只涉及 218 笔订单？找出同时触发多条规则的订单。
orders_v1.loc[
    orders_v1['review_rule_count'].gt(1),
    ['order_id', 'order_status'] + REVIEW_FLAG_COLS
]

,order_id,order_status,flag_delivered_missing_date,flag_delivered_missing_approval,flag_delivered_missing_carrier,flag_carrier_before_purchase,flag_delivery_before_carrier,flag_canceled_has_delivery_date
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,True,False,True,False,False,False


In [8]:
# summarize_metrics 返回字典，保存整数计数和未四舍五入的比率。
metrics = summarize_metrics(orders_v1)
print('运输有效样本：', metrics['valid_transit_orders'])
print('平均运输天数：{:.2f}'.format(metrics['transit_mean_days']))
print('运输 P50：{:.2f}'.format(metrics['transit_p50_days']))
print('运输 P90：{:.2f}'.format(metrics['transit_p90_days']))
print('延迟有效样本：', metrics['valid_delay_orders'])
print('延迟订单数：', metrics['late_orders'])
print('延迟率：{:.2%}'.format(metrics['late_rate']))

运输有效样本： 96281
平均运输天数：9.33
运输 P50：7.10
运输 P90：18.91
延迟有效样本： 96470
延迟订单数： 6534
延迟率：6.77%


## 保存结果

配套脚本会重新读取原始文件、调用同一函数，再保存整理表、待核查清单、规则汇总和指标。可在项目目录的终端执行：

```powershell
python .\src\02_clean_orders.py
```

输出和规则说明见 `docs/02_订单清洗整理.md`。CSV 不保存日期类型，Parquet 将在后面的环境配置步骤补充。

## 我的理解

1. 为什么函数内部用 `raw_orders.copy()`？

我的回答：创建原始 DataFrame 的工作副本，使函数中的类型转换、新增字段等操作不会修改传入的 raw_orders。

2. `orders_v1 = clean_orders(raw_orders)` 中，输入是什么，返回值是什么？

我的回答：输入是原始订单表 raw_orders。返回值是整理后的新 DataFrame，保留全部订单及原有字段，转换时间类型，并增加六类核查标记、有效样本标记、运输耗时、延迟天数等字段。orders_v1 用来接收这张新表。

3. 为什么不能把 `flag_needs_review` 为 True 的订单全部删除？

我的回答：flag_needs_review=True 表示订单触发了至少一条核查规则，但其他信息仍可能有效。不同指标依赖不同字段，应保留订单和标记，分别筛选各指标的有效样本，并报告排除数量。统一删除会损失可用信息，也可能改变样本结构，影响分析结果。

In [9]:
def add_time_order_flag(df, earlier_col, later_col, flag_name):
    result = df.copy()

    can_compare = (
        result[earlier_col].notna()
        & result[later_col].notna()
    )

    result[flag_name] = (
        can_compare
        & (result[later_col] < result[earlier_col])
    )

    return result

In [ ]:
orders_v1 = add_time_order_flag(
    orders_v1,
    'order_purchase_timestamp',
    'order_approved_at',
    'flag_approval_before_purchase'
)

orders_v1 = add_time_order_flag(
    orders_v1,
    'order_approved_at',
    'order_delivered_carrier_date',
    'flag_carrier_before_approval'
)

In [11]:
orders_v1[
    [
        'flag_approval_before_purchase',
        'flag_carrier_before_approval'
    ]
].sum()

flag_approval_before_purchase       0
flag_carrier_before_approval     1359
dtype: int64

In [12]:
orders_v1.loc[
    orders_v1['flag_carrier_before_approval'],
    [
        'order_id',
        'order_purchase_timestamp',
        'order_approved_at',
        'order_delivered_carrier_date'
    ]
].head(3)

,order_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date
15,dcb36b511fcac050b97cd5c05de84dc3,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00
64,688052146432ef8253587b930b01a06d,2018-04-22 08:48:13,2018-04-24 18:25:22,2018-04-23 19:19:14
199,58d4c4747ee059eeeb865b349b41f53a,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-24 12:57:00
